# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import math
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform, IMAGENET_MEAN, IMAGENET_STD
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import multi_attr_balanced_sampler
from src.losses.imbalanced import FocalLoss
from src.augment.mix import cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]
# 각 실험마다 RUN_NAME 만 바꿔서 동일 프로젝트에 누적하세요.
EXPERIMENT_NAME = "ir-focal+ir-sampler+randaug+cutmix"

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
import os, sys, zipfile, subprocess
from torchvision import transforms

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# RandAugment 포함 train transform (level3 전용)
def train_transform_l3(img_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform_l3())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 속성별 IR (Imbalance Ratio) 자동 계산
IR = {}
for a in ATTRIBUTES:
    c = train_ds.class_counts(a).float()
    IR[a] = (c.max() / c[c > 0].min()).item()
print("Imbalance Ratio:", {a: round(v, 1) for a, v in IR.items()})

# 3속성 IR 통합 multi-attribute balanced sampler
sampler = multi_attr_balanced_sampler(train_ds, IR)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=2, pin_memory=True)

for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

In [ ]:
# IR 기반 gamma 자동 계산 (불균형 클수록 gamma 높게)
gamma_dict = {a: round(1.0 + math.log(IR[a]), 1) for a in ATTRIBUTES}
print(f"FocalLoss gamma — {gamma_dict}")

# 속성별 FocalLoss (IR로 자동 결정된 gamma)
loss_fns = {a: FocalLoss(gamma=gamma_dict[a]).to(device) for a in ATTRIBUTES}

# IR 정규화 가중치 — mixed_loss 및 wandb 로깅용
ir_total    = sum(IR.values())
mix_weights = {a: IR[a] / ir_total for a in ATTRIBUTES}

set_seed(SEED, deterministic=True)
model  = vit_small_patch16_224().to(device)
epochs = 30
optim  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "backbone": "vit_s16",
        "sampler": "multi_attr_ir_balanced",
        "loss": {a: f"focal_g{gamma_dict[a]}" for a in ATTRIBUTES},
        "augmentation": "randaugment(n=2,m=9)+cutmix(a=1.0)",
        "IR": {a: round(v, 1) for a, v in IR.items()},
        "mix_weights": {a: round(v, 3) for a, v in mix_weights.items()},
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME],
)
trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)

In [ ]:
from tqdm import tqdm

def step_with_mix(images, targets):
    """CutMix + IR 기반 속성별 가중 loss."""
    x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam, weights=mix_weights)


history = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

for epoch in range(epochs):
    model.train()
    running, n_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f"train e{epoch+1:02d}", leave=False)
    for batch in pbar:
        images  = batch["image"].to(device, non_blocking=True)
        targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

        optim.zero_grad(set_to_none=True)
        loss = step_with_mix(images, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

        running   += loss.item()
        n_batches += 1
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss  = running / max(n_batches, 1)
    val_metrics = trainer.evaluate(val_loader)
    sched.step()

    history["train_loss"].append(train_loss)
    history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
    history["val_per_mf1"].append(val_metrics["per_macro_f1"])

    print(
        f"[epoch {epoch+1:02d}/{epochs}] "
        f"loss={train_loss:.4f}  "
        f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
        f"per={val_metrics['per_macro_f1']}"
    )

    log_payload = {
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "val/avg_macro_f1": val_metrics["avg_macro_f1"],
        "lr": optim.param_groups[0]["lr"],
    }
    for a, v in val_metrics["per_macro_f1"].items():
        log_payload[f"val/mf1_{a}"] = v
    logger.log(log_payload, step=epoch)

os.makedirs("../checkpoints", exist_ok=True)
torch.save({"state_dict": model.state_dict(), "history": history},
           f"../checkpoints/level3_{EXPERIMENT_NAME}.pth")
print(f"저장 완료 → ../checkpoints/level3_{EXPERIMENT_NAME}.pth")

In [ ]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

In [ ]:
# ── Ablation B: IR sampler only (FocalLoss X, RandAugment X, CutMix X) ──────
EXPERIMENT_NAME_B = "ir-sampler-only"

# 기본 augmentation (RandAugment 없음)
train_ds_b     = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
sampler_b      = multi_attr_balanced_sampler(train_ds_b, IR)
train_loader_b = DataLoader(train_ds_b, batch_size=BATCH, sampler=sampler_b,
                            num_workers=2, pin_memory=True)

# CE loss (FocalLoss 없음)
loss_fns_b = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

set_seed(SEED, deterministic=True)
model_b = vit_small_patch16_224().to(device)
optim_b = torch.optim.AdamW(model_b.parameters(), lr=5e-4, weight_decay=5e-2)
sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(optim_b, T_max=epochs)

logger_b = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME_B}",
    config={
        "backbone": "vit_s16",
        "sampler": "multi_attr_ir_balanced",
        "loss": "cross_entropy",
        "augmentation": "standard",
        "IR": {a: round(v, 1) for a, v in IR.items()},
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME_B],
)
trainer_b = MultiTaskTrainer(model_b, optim_b, sched_b, loss_fns_b, device,
                             TrainConfig(epochs=epochs), logger=logger_b)

In [ ]:
from tqdm import tqdm

history_b = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

for epoch in range(epochs):
    model_b.train()
    running, n_batches = 0.0, 0

    pbar = tqdm(train_loader_b, desc=f"train e{epoch+1:02d}", leave=False)
    for batch in pbar:
        images  = batch["image"].to(device, non_blocking=True)
        targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

        optim_b.zero_grad(set_to_none=True)
        logits = model_b(images)
        loss   = sum(loss_fns_b[a](logits[a], targets[a]) for a in ATTRIBUTES)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
        optim_b.step()

        running   += loss.item()
        n_batches += 1
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss  = running / max(n_batches, 1)
    val_metrics = trainer_b.evaluate(val_loader)
    sched_b.step()

    history_b["train_loss"].append(train_loss)
    history_b["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
    history_b["val_per_mf1"].append(val_metrics["per_macro_f1"])

    print(
        f"[epoch {epoch+1:02d}/{epochs}] "
        f"loss={train_loss:.4f}  "
        f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
        f"per={val_metrics['per_macro_f1']}"
    )

    log_payload = {
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "val/avg_macro_f1": val_metrics["avg_macro_f1"],
        "lr": optim_b.param_groups[0]["lr"],
    }
    for a, v in val_metrics["per_macro_f1"].items():
        log_payload[f"val/mf1_{a}"] = v
    logger_b.log(log_payload, step=epoch)

torch.save({"state_dict": model_b.state_dict(), "history": history_b},
           f"../checkpoints/level3_{EXPERIMENT_NAME_B}.pth")
print(f"저장 완료 → ../checkpoints/level3_{EXPERIMENT_NAME_B}.pth")

In [ ]:
val_pred_b, _, val_tgt_b, _ = collect_predictions(model_b, val_loader, device)
cms_b = confusion_matrices(val_pred_b, val_tgt_b)
prf_b = per_class_prf(val_pred_b, val_tgt_b)
for a in ATTRIBUTES:
    logger_b.log_confusion_matrix(f"final/cm_{a}", cms_b[a], CLASS_NAMES[a])
    rows = list(zip(prf_b[a]["class"], prf_b[a]["precision"], prf_b[a]["recall"],
                    prf_b[a]["f1"], prf_b[a]["support"]))
    logger_b.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"],
                       [list(r) for r in rows])
logger_b.finish()

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.